# Phase 1: CIFAR-10 and ResNet-56 Baseline

This notebook prepares the first executable ML slice of the CNN Compression Toolkit. It loads CIFAR-10, creates deterministic train/validation/calibration/test splits, defines a CIFAR-10 ResNet-56, optionally trains a checkpoint, validates a raw `state_dict`, preserves independent reference and working models, evaluates M0 metrics, and writes reproducible artifacts.

ONNX export, latency benchmarking, pruning, BN recovery, distillation, and quantization are intentionally deferred to later phases. Test data is used only for the final baseline evaluation.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import os
import platform
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/home/avinash/Projects/CNN Compression Toolkit')
DATA_ROOT = PROJECT_ROOT / 'data' / 'cifar10'
CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'resnet56_cifar10.pt'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'runs'
SEED = 20261024
NUM_CLASSES = 10
INPUT_SHAPE = (1, 3, 32, 32)
CALIBRATION_SIZE = 512
VALIDATION_SIZE = 5000
BATCH_SIZE = 128
NUM_WORKERS = 2
DOWNLOAD_CIFAR10 = True
TRAIN_IF_CHECKPOINT_MISSING = True
TRAIN_EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Project root:', PROJECT_ROOT)

Python: 3.11.17
PyTorch: 2.14.1+cu130
Device: cpu
Project root: /home/avinash/Projects/CNN Compression Toolkit


## 1. CIFAR-10 data and deterministic split policy

The validation and calibration subsets are selected only from the 50,000-image training split. The test split remains untouched until baseline evaluation.

In [2]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_augmented = datasets.CIFAR10(DATA_ROOT, train=True, download=DOWNLOAD_CIFAR10, transform=train_transform)
train_eval = datasets.CIFAR10(DATA_ROOT, train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(DATA_ROOT, train=False, download=DOWNLOAD_CIFAR10, transform=eval_transform)

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_eval), generator=generator).tolist()
validation_indices = permutation[:VALIDATION_SIZE]
calibration_start = VALIDATION_SIZE
calibration_indices = permutation[calibration_start:calibration_start + CALIBRATION_SIZE]
train_indices = permutation[calibration_start + CALIBRATION_SIZE:]
assert len(train_indices) + len(validation_indices) + len(calibration_indices) == 50000
assert set(train_indices).isdisjoint(validation_indices)
assert set(train_indices).isdisjoint(calibration_indices)
assert set(validation_indices).isdisjoint(calibration_indices)

train_loader = DataLoader(Subset(train_augmented, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
validation_loader = DataLoader(Subset(train_eval, validation_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
calibration_loader = DataLoader(Subset(train_eval, calibration_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'train={len(train_indices)}, validation={len(validation_indices)}, calibration={len(calibration_indices)}, test={len(test_dataset)}')

train=44488, validation=5000, calibration=512, test=10000


In [3]:
split_metadata = {
    'seed': SEED,
    'calibration_size': CALIBRATION_SIZE,
    'validation_size': VALIDATION_SIZE,
    'train_indices': train_indices,
    'validation_indices': validation_indices,
    'calibration_indices': calibration_indices,
    'test_size': len(test_dataset),
}
print('Calibration examples:', len(split_metadata['calibration_indices']))
print('Test examples reserved for final evaluation:', len(test_dataset))

Calibration examples: 512
Test examples reserved for final evaluation: 10000


## 2. Canonical CIFAR-10 ResNet-56

In [4]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.relu(x + identity)

class CIFARResNet(nn.Module):
    def __init__(self, depth: int = 56, num_classes: int = 10):
        super().__init__()
        if (depth - 2) % 6 != 0:
            raise ValueError('CIFAR ResNet depth must satisfy (depth - 2) % 6 == 0')
        blocks_per_stage = (depth - 2) // 6
        self.in_channels = 16
        self.conv1 = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_stage(16, blocks_per_stage, 1)
        self.layer2 = self._make_stage(32, blocks_per_stage, 2)
        self.layer3 = self._make_stage(64, blocks_per_stage, 2)
        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64, num_classes)

    def _make_stage(self, channels: int, blocks: int, stride: int) -> nn.Sequential:
        strides = [stride] + [1] * (blocks - 1)
        layers = []
        for current_stride in strides:
            layers.append(BasicBlock(self.in_channels, channels, current_stride))
            self.in_channels = channels
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        x = self.avgpool(x)
        return self.fc(torch.flatten(x, 1))

model = CIFARResNet(depth=56, num_classes=NUM_CLASSES).to(DEVICE)
with torch.no_grad():
    shape_check = model(torch.zeros(INPUT_SHAPE, device=DEVICE))
assert tuple(shape_check.shape) == (1, NUM_CLASSES)
assert torch.isfinite(shape_check).all()
print('ResNet-56 forward shape:', tuple(shape_check.shape))
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))

ResNet-56 forward shape: (1, 10)
Parameters: 855770


## 3. Checkpoint loading or optional training

The checkpoint format is a raw `state_dict` saved with `torch.save(model.state_dict(), path)`. Training is disabled by default. Enable `TRAIN_IF_CHECKPOINT_MISSING` only when you want this notebook to train a checkpoint.

In [5]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


def evaluate_validation(network: nn.Module, loader: DataLoader) -> tuple[float, float]:
    network.eval()
    criterion = nn.CrossEntropyLoss()
    correct = 0
    total = 0
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = network(images)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            correct += int((logits.argmax(dim=1) == labels).sum())
            total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


def run_epoch(network: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> float:
    training = optimizer is not None
    network.train(training)
    total_loss = 0.0
    total = 0
    criterion = nn.CrossEntropyLoss()
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = network(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)
    return total_loss / max(total, 1)


if not CHECKPOINT_PATH.exists() and TRAIN_IF_CHECKPOINT_MISSING:
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)
    best_validation_accuracy = float('-inf')
    best_validation_loss = float('inf')
    best_epoch = None
    best_state = None
    history = []
    for epoch in range(1, TRAIN_EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimizer)
        validation_loss, validation_accuracy = evaluate_validation(model, validation_loader)
        scheduler.step()
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'validation_loss': validation_loss,
            'validation_accuracy': validation_accuracy,
        })
        if validation_accuracy > best_validation_accuracy:
            best_validation_accuracy = validation_accuracy
            best_validation_loss = validation_loss
            best_epoch = epoch
            best_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
        print(
            f'epoch {epoch}/{TRAIN_EPOCHS}: train_loss={train_loss:.4f}, '
            f'validation_loss={validation_loss:.4f}, validation_accuracy={validation_accuracy:.4f}'
        )
    if best_state is None:
        raise RuntimeError('Training completed without selecting a checkpoint')
    model.load_state_dict(best_state, strict=True)
    torch.save(best_state, CHECKPOINT_PATH)
    print(
        f'Saved best checkpoint from epoch {best_epoch}: '
        f'validation_accuracy={best_validation_accuracy:.4f}, '
        f'validation_loss={best_validation_loss:.4f}'
    )
elif not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'Missing {CHECKPOINT_PATH}. Provision a trained checkpoint or set TRAIN_IF_CHECKPOINT_MISSING=True.'
    )

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
if not isinstance(checkpoint, dict) or not all(isinstance(value, torch.Tensor) for value in checkpoint.values()):
    raise TypeError('Checkpoint must be a raw state_dict mapping parameter names to tensors')
model.load_state_dict(checkpoint, strict=True)
model.eval()
checkpoint_bytes = CHECKPOINT_PATH.read_bytes()
checkpoint_sha256 = hashlib.sha256(checkpoint_bytes).hexdigest()
print('Loaded checkpoint:', CHECKPOINT_PATH)
print('SHA-256:', checkpoint_sha256)

Loaded checkpoint: /home/avinash/Projects/CNN Compression Toolkit/checkpoints/resnet56_cifar10.pt
SHA-256: 938b75be4e54ad397a699cf7a827f11779486add1c7a4cf094c9467ccdb0fc82


In [6]:
reference_model = copy.deepcopy(model).to(DEVICE).eval()
working_model = copy.deepcopy(model).to(DEVICE).eval()
for parameter in reference_model.parameters():
    parameter.requires_grad_(False)

assert reference_model is not working_model
reference_before = {name: tensor.detach().clone() for name, tensor in reference_model.state_dict().items()}
with torch.no_grad():
    next(working_model.parameters()).add_(1.0)
assert all(torch.equal(reference_model.state_dict()[name], tensor) for name, tensor in reference_before.items())
print('Reference and working models are independent.')

Reference and working models are independent.


## 4. M0 baseline metrics

In [7]:
def evaluate_classifier(network: nn.Module, loader: DataLoader) -> dict:
    network.eval()
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    total_loss = 0.0
    total = 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, labels in loader:
            logits = network(images.to(DEVICE))
            labels = labels.to(DEVICE)
            predictions = logits.argmax(dim=1)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            total += labels.size(0)
            for actual, predicted in zip(labels.cpu(), predictions.cpu()):
                confusion[int(actual), int(predicted)] += 1
    true_positive = confusion.diag().float()
    precision_denominator = confusion.sum(dim=0).float()
    recall_denominator = confusion.sum(dim=1).float()
    precision = torch.where(precision_denominator > 0, true_positive / precision_denominator, torch.zeros_like(true_positive))
    recall = torch.where(recall_denominator > 0, true_positive / recall_denominator, torch.zeros_like(true_positive))
    f1_denominator = precision + recall
    f1 = torch.where(f1_denominator > 0, 2 * precision * recall / f1_denominator, torch.zeros_like(precision))
    return {
        'loss': total_loss / max(total, 1),
        'accuracy': float(true_positive.sum() / max(total, 1)),
        'macro_precision': float(precision.mean()),
        'macro_recall': float(recall.mean()),
        'macro_f1': float(f1.mean()),
        'confusion_matrix': confusion.tolist(),
        'sample_count': total,
    }

baseline_metrics = evaluate_classifier(reference_model, test_loader)
baseline_metrics.update({
    'architecture': 'resnet56',
    'checkpoint_sha256': checkpoint_sha256,
    'checkpoint_format': 'state_dict',
    'input_shape': list(INPUT_SHAPE),
    'num_classes': NUM_CLASSES,
    'device': str(DEVICE),
    'normalization_mean': list(MEAN),
    'normalization_std': list(STD),
})
print(json.dumps({key: value for key, value in baseline_metrics.items() if key != 'confusion_matrix'}, indent=2))

{
  "loss": 0.5042425422430038,
  "accuracy": 0.8285999894142151,
  "macro_precision": 0.8274043798446655,
  "macro_recall": 0.8286000490188599,
  "macro_f1": 0.8274906873703003,
  "sample_count": 10000,
  "architecture": "resnet56",
  "checkpoint_sha256": "938b75be4e54ad397a699cf7a827f11779486add1c7a4cf094c9467ccdb0fc82",
  "checkpoint_format": "state_dict",
  "input_shape": [
    1,
    3,
    32,
    32
  ],
  "num_classes": 10,
  "device": "cpu",
  "normalization_mean": [
    0.4914,
    0.4822,
    0.4465
  ],
  "normalization_std": [
    0.247,
    0.2435,
    0.2616
  ]
}


In [8]:
experiment_payload = {
    'architecture': 'resnet56',
    'seed': SEED,
    'checkpoint_sha256': checkpoint_sha256,
    'calibration_size': CALIBRATION_SIZE,
    'validation_size': VALIDATION_SIZE,
}
config_hash = hashlib.sha256(json.dumps(experiment_payload, sort_keys=True).encode()).hexdigest()[:8]
experiment_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + config_hash
run_root = ARTIFACT_ROOT / experiment_id
run_root.mkdir(parents=True, exist_ok=True)

config_record = {
    **experiment_payload,
    'project_root': str(PROJECT_ROOT),
    'data_root': str(DATA_ROOT),
    'checkpoint_path': str(CHECKPOINT_PATH),
    'input_shape': list(INPUT_SHAPE),
    'normalization_mean': list(MEAN),
    'normalization_std': list(STD),
}
(run_root / 'config.json').write_text(json.dumps(config_record, indent=2) + '\n', encoding='utf-8')
(run_root / 'split_indices.json').write_text(json.dumps(split_metadata, indent=2) + '\n', encoding='utf-8')
(run_root / 'checkpoint.json').write_text(json.dumps({
    'path': str(CHECKPOINT_PATH),
    'format': 'state_dict',
    'sha256': checkpoint_sha256,
    'architecture': 'resnet56',
}, indent=2) + '\n', encoding='utf-8')
(run_root / 'baseline_metrics.json').write_text(json.dumps(baseline_metrics, indent=2) + '\n', encoding='utf-8')
(run_root / 'environment.json').write_text(json.dumps({
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'numpy': np.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
}, indent=2) + '\n', encoding='utf-8')
print('Artifacts written to:', run_root)

Artifacts written to: /home/avinash/Projects/CNN Compression Toolkit/artifacts/runs/20261025T175254Z-a2f75753


### VGG-16-BN status

This notebook intentionally targets ResNet-56 first. Add VGG-16-BN only after the ResNet-56 data, checkpoint, model-isolation, and M0 baseline checks pass and a compatible VGG-16-BN `state_dict` checkpoint is available.